# Install Dependency

In [1]:
!pip install transformers torch accelerate tqdm

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
import pandas as pd
import torch

from transformers import pipeline
from tqdm import tqdm

In [4]:
device = 0 if torch.cuda.is_available() else -1

print(
    f"Device digunakan: {'GPU' if device == 0 else 'CPU'}"
)

Device digunakan: GPU


In [5]:
sentiment_pipeline = pipeline(
    task="text-classification",
    model="mdhugol/indonesia-bert-sentiment-classification",
    tokenizer="mdhugol/indonesia-bert-sentiment-classification",
    device=device,
    truncation=True,
    max_length=512
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/KaburAjaDulu/data/cleaned_data.csv")
print(f"Total data: {len(df)}")

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total data: 18814


,created_at,full_text,id_str,platform,clean_text
0,Tue Sep 30 23:48:17 +0000 2025,EMANG BNER GUE #KaburAjaDulu ATP,1973173082253828399,X,emang bner saya kaburajadulu atp
1,Tue Sep 30 19:41:13 +0000 2025,Wkwkwkw masih aja. Saran saya saatnya #KaburAj...,1973110905211855020,X,wkwkwkw masih saja saran saya saatnya kaburaja...
2,Tue Sep 30 15:25:47 +0000 2025,Selamat dan sukses kepada siswa-siswi LPK Yosh...,1973046626290897319,X,selamat dan sukses kepada siswa siswi lpk yosh...
3,Tue Sep 30 14:43:04 +0000 2025,@alestarbluu Sumpahh aku pro ke statement #kab...,1973035876054905276,X,sumpahh aku profesional ke statement kaburajad...
4,Tue Sep 30 12:47:18 +0000 2025,Manual order shipped safely️ bukunya #KaburAja...,1973006742117298281,X,manual order shipped safely bukunya kaburajadu...


In [9]:
LABEL_MAP = {
    "LABEL_0": "positif",
    "LABEL_1": "netral",
    "LABEL_2": "negatif",
}

texts = df["clean_text"].tolist()
results = []
batch_size = 32

In [10]:
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    try:
        outputs = sentiment_pipeline(
            batch,
            truncation=True,
            max_length=512
        )
        for out in outputs:
            results.append({
                "sentiment": LABEL_MAP.get(
                    out["label"],
                    out["label"].lower()
                ),
                "confidence": round(
                    out["score"] * 100,
                    2
                )
            })
    except Exception as e:
        print(f"Error batch {i}: {e}")
        for _ in batch:
            results.append({
                "sentiment": "netral",
                "confidence": 0.0
            })

100%|██████████| 588/588 [03:05<00:00,  3.18it/s]


In [11]:
df["sentiment"] = [
    r["sentiment"]
    for r in results
]
df["confidence"] = [
    r["confidence"]
    for r in results
]
df.head()

,created_at,full_text,id_str,platform,clean_text,sentiment,confidence
0,Tue Sep 30 23:48:17 +0000 2025,EMANG BNER GUE #KaburAjaDulu ATP,1973173082253828399,X,emang bner saya kaburajadulu atp,negatif,99.65
1,Tue Sep 30 19:41:13 +0000 2025,Wkwkwkw masih aja. Saran saya saatnya #KaburAj...,1973110905211855020,X,wkwkwkw masih saja saran saya saatnya kaburaja...,netral,95.98
2,Tue Sep 30 15:25:47 +0000 2025,Selamat dan sukses kepada siswa-siswi LPK Yosh...,1973046626290897319,X,selamat dan sukses kepada siswa siswi lpk yosh...,positif,79.15
3,Tue Sep 30 14:43:04 +0000 2025,@alestarbluu Sumpahh aku pro ke statement #kab...,1973035876054905276,X,sumpahh aku profesional ke statement kaburajad...,negatif,99.58
4,Tue Sep 30 12:47:18 +0000 2025,Manual order shipped safely️ bukunya #KaburAja...,1973006742117298281,X,manual order shipped safely bukunya kaburajadu...,positif,81.90


In [12]:
df["sentiment"].value_counts()

,count
sentiment,
negatif,11411
positif,3994
netral,3409


In [13]:
df["sentiment"].value_counts(normalize=True) * 100

,proportion
sentiment,
negatif,60.651642
positif,21.228872
netral,18.119485


In [14]:
df.groupby(
    ["platform", "sentiment"]
).size()

platform  sentiment
TikTok    negatif      5632
          netral       2217
          positif      2513
X         negatif      5779
          netral       1192
          positif      1481
dtype: int64

In [15]:
df.to_csv(
    "/content/sentiment_results.csv",
    index=False
)